<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings and methodology questions

**Finding 1 — The Anatomy of Growing Content**

The paper reports that growing content is longer and younger than declining content: growing pages average about 3.2K words and 184 days old, while declining pages average about 2.3K words and 230 days old. The growth/decline label comes from the paper's trend-direction definition, which is based on the change in impressions between the latest 30 days and the previous 30 days. The paper correctly treats this as an observational comparison rather than proof that adding words or reducing age causes growth.

My methodology question is: would the finding remain similar if the comparison controlled for client, search position, topic, and existing traffic level? Pages from the same client may have different publishing and measurement patterns, so a grouped or matched validation design could make the comparison more robust.


**Finding 2 — The Freshness Multiplier**

The paper reports that the 31-90 day freshness window has the strongest stable growth-to-decline ratio at 7.88:1. It also reports that 365+ day pages refreshed within 30 days had much higher health and impressions than older unrefreshed pages. The growth/decline outcome is based on trend direction, while health and impressions are measured performance outcomes. The paper notes that the 361+ freshness bucket is unstable because it contains very few declining pages.

My methodology question is: does the refresh comparison separate the effect of refreshing a page from differences in page quality, historical visibility, topic, and demand? A matched before/after or controlled comparison would provide stronger evidence about whether refreshing itself contributes to the observed improvement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation design

In Week 5, the model was evaluated using a row-based split. This can be optimistic when multiple pages from the same client are present because the model may learn client-specific patterns.

For this validation audit, I will use a client-grouped split so that pages from the same client do not appear in both training and testing. This gives a more honest estimate of how the model performs on clients it has not seen during training.

The feature window remains February 2026 and the outcome window remains March 2026. The Week-5 row-split result is reported as the "before" result, while the client-grouped result is the "after" result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Starting honest validation audit...")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# 1. Warehouse setup
# ---------------------------------------------------------

if "con" not in globals():
    %pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

    import duckdb
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is not available in Colab Secrets.")

    con = duckdb.connect()

    # Keep the Hugging Face token out of SQL text.
    con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
    con.execute("""
        CREATE OR REPLACE SECRET hf_secret (
            TYPE HUGGINGFACE,
            TOKEN getvariable('hf_token')
        )
    """)

    FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
    FEB = f"{FACT}/month=2026-02/*.parquet"
    MAR = f"{FACT}/month=2026-03/*.parquet"

# ---------------------------------------------------------
# 2. February features
#    Keep CLIENT + CONTENT so we can group by client.
# ---------------------------------------------------------

feb_agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(gsc_avg_position) AS feb_avg_position,
        SUM(ga4_pageviews) AS feb_pageviews,
        SUM(ga4_sessions) AS feb_sessions,
        SUM(ga4_users) AS feb_users,
        SUM(ga4_engaged_sessions) AS feb_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS feb_engagement_sec,
        SUM(sessions_organic) AS feb_organic_sessions,
        SUM(sessions_ai) AS feb_ai_sessions

    FROM read_parquet('{FEB}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# ---------------------------------------------------------
# 3. March outcome
# ---------------------------------------------------------

mar_agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS mar_clicks

    FROM read_parquet('{MAR}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# ---------------------------------------------------------
# 4. Join
# ---------------------------------------------------------

model_df = feb_agg.merge(
    mar_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df = model_df.dropna(subset=["mar_clicks"]).copy()

# Outcome:
# 1 = March clicks lower than February clicks
# 0 = otherwise
model_df["decline_label"] = (
    model_df["mar_clicks"] < model_df["feb_clicks"]
).astype(int)

print("Rows after join:", len(model_df))
print("Number of clients:", model_df["client_hash_id"].nunique())

# ---------------------------------------------------------
# 5. Final February-only feature set
# ---------------------------------------------------------

feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_pageviews",
    "feb_sessions",
    "feb_users",
    "feb_engaged_sessions",
    "feb_engagement_sec",
    "feb_organic_sessions",
    "feb_ai_sessions"
]

X = (
    model_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = model_df["decline_label"]
groups = model_df["client_hash_id"]

# ---------------------------------------------------------
# 6. Honest client-grouped split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("\nGrouped split:")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

# ---------------------------------------------------------
# 7. Train model
# ---------------------------------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

after_accuracy = accuracy_score(
    y_test,
    predictions
)

# ---------------------------------------------------------
# 8. Base rate
# ---------------------------------------------------------

base_rate = y_test.mean()

print("\nValidation results")
print("------------------")
print("Week-5 row-split accuracy (before): 0.9627")
print("Client-grouped accuracy (after):", round(after_accuracy, 4))
print("Test-set decline base rate:", round(base_rate, 4))

print("\nInterpretation:")
if after_accuracy < 0.9627:
    print("Accuracy decreased under client-grouped validation.")
    print("This suggests the original row-split result was optimistic.")
else:
    print("Accuracy did not decrease under client-grouped validation.")

if len(train_clients & test_clients) == 0:
    print("Grouped validation check: PASSED")
else:
    print("Grouped validation check: FAILED")

Starting honest validation audit...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after join: 303572
Number of clients: 50

Grouped split:
Training rows: 264134
Testing rows: 39438
Training clients: 40
Testing clients: 10
Client overlap: 0

Validation results
------------------
Week-5 row-split accuracy (before): 0.9627
Client-grouped accuracy (after): 0.8782
Test-set decline base rate: 0.1206

Interpretation:
Accuracy decreased under client-grouped validation.
This suggests the original row-split result was optimistic.
Grouped validation check: PASSED


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final feature set for label-derived fields, future-window fields, and other decision-derived signals. The final model features are constructed only from February 2026 data.

March 2026 clicks are used to create the decline outcome, but they are not included as model features. Client and content identifiers are used for joining and client-grouped validation, not as predictive features.

I also performed a deliberate leakage test by adding March clicks as a suspect feature. This feature is excluded from the final model because it comes from the future outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABO

print("Starting leakage audit...")

# ---------------------------------------------------------
# 1. Check final feature set
# ---------------------------------------------------------

leakage_keywords = [
    "mar_",
    "march",
    "trend",
    "label",
    "decline",
    "future",
    "flag",
    "action",
    "priority"
]

suspect_features = [
    col for col in feature_cols
    if any(word in col.lower() for word in leakage_keywords)
]

print("Final feature set:")
for col in feature_cols:
    print(" -", col)

print("\nPotential leakage names found:")
print(suspect_features if suspect_features else "None")

# ---------------------------------------------------------
# 2. Direct leakage checks
# ---------------------------------------------------------

assert "mar_clicks" not in feature_cols
assert "decline_label" not in feature_cols
assert all(not col.startswith("mar_") for col in feature_cols)

print("\nDirect leakage checks: PASSED")
print("March outcome is not present in the final feature vector.")

# ---------------------------------------------------------
# 3. Deliberate leakage experiment
# ---------------------------------------------------------

X_leaky = X.copy()

# Deliberately add the future outcome as a suspect feature.
X_leaky["LEAKY_MARCH_CLICKS"] = model_df["mar_clicks"].values

leaky_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

leaky_model.fit(
    X_leaky.iloc[train_idx],
    y.iloc[train_idx]
)

leaky_predictions = leaky_model.predict(
    X_leaky.iloc[test_idx]
)

leaky_accuracy = accuracy_score(
    y.iloc[test_idx],
    leaky_predictions
)

print("\nDeliberate leakage experiment:")
print(
    "Accuracy with future March clicks:",
    round(leaky_accuracy, 4)
)

print("\nFinal decision:")
print("The deliberately leaky feature is excluded from the final model.")
print("The production feature vector remains February-only.")


Starting leakage audit...
Final feature set:
 - feb_impressions
 - feb_clicks
 - feb_avg_position
 - feb_pageviews
 - feb_sessions
 - feb_users
 - feb_engaged_sessions
 - feb_engagement_sec
 - feb_organic_sessions
 - feb_ai_sessions

Potential leakage names found:
None

Direct leakage checks: PASSED
March outcome is not present in the final feature vector.

Deliberate leakage experiment:
Accuracy with future March clicks: 0.9998

Final decision:
The deliberately leaky feature is excluded from the final model.
The production feature vector remains February-only.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Safe research claim

**Original bold claim:**  
The Logistic Regression model accurately predicts which pages will decline and beats the baseline.

**Rewritten claim:**  
On this dataset, the Logistic Regression model achieved 0.9627 accuracy under the Week-5 row split and 0.8782 accuracy under client-grouped validation. The grouped result was lower, so the row-split result appears optimistic. The model can be used as a directional decision-support signal for prioritizing content for review, but it should not be treated as proof that a page will decline or that refreshing it will improve search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.